# 02 — The model does not call the tool

Module 01 ended with a home-made line: `COUNT helena`. We parsed it. We ran a function. The model never touched the shop.

The official API is the same idea with a real field. The model emits a **name** and a **JSON string**. Your code decides whether to run anything. That is the whole module.

If you remember one picture, remember this one:

**Assumed**

```mermaid
flowchart LR
    A["user"] --> B["software"]
    B --> C["LLM"]
    C --> D["tools"]
```

**Actual**

```mermaid
flowchart LR
    A["user"] --> B["software"]
    B --> C["LLM"]
    C -->|"emits a name and a JSON string"| B
    B -->|"your code decides"| D["tools"]
```

Security, sandboxing, and the loop in module 03 only make sense after this. Prompt injection is not the model doing something. It is your code doing something because the model asked.


## 1. Learn

You already know the object. In module 00, `message` had `role` and `content`. Today the same `message` can also have `tool_calls`. `content` is often empty. `finish_reason` is `tool_calls` instead of `stop`.

```mermaid
flowchart TD
    A["your code"] -->|"create(messages, tools=[...])"| B["LLM"]
    B -->|"returns"| C["message.tool_calls[0]"]
    C --> D["name: get_fact"]
    C --> E["arguments: {'city': 'Amsterdam'} — a string"]
    C --> F["id: call_..."]
    C --> G{"your code decides"}
    G -->|"ignore it"| H["nothing runs"]
    G -->|"act on it"| I["json.loads(arguments)"]
    I --> J["get_fact('Amsterdam')"]
    J --> K["append a tool message, with that id"]
    K --> L["LLM"]
    L -->|"now it can talk"| M["message.content"]
```

`arguments` is a string, not a dict. You parse it. The model can emit bad JSON or a city you do not have. That is why your code is in the middle.

A single reply can carry more than one tool call. We will run those one after another. Module 07 is when we run them at the same time.

The data today is local: flights and city facts in two CSV files. A live weather API would close the Prague hole from module 00 with the same pattern. We stay on disk so this room does not depend on a second website. The music shop database is still later.

After the hand-written loop we take a first look at the OpenAI Agents SDK: an agent with no tools, a trace, the decorator that writes the JSON you typed, then a session so two runs can share memory. Module 08 is the full SDK.


## 2. Do

### Load the environment and the two files


In [1]:
from pathlib import Path
import csv
import json
import os

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = os.environ.get("MODEL_DEFAULT", "").strip()
assert api_key, "OPENAI_API_KEY is missing. Copy .env.example to .env and add the class key."
assert model, "MODEL_DEFAULT is missing from .env."

client = OpenAI()

facts_rows = list(csv.DictReader(open(ROOT / "data" / "fun_facts.csv")))
flight_rows = list(csv.DictReader(open(ROOT / "data" / "flight_data.csv")))
print("OPENAI_API_KEY is set:", True)
print("MODEL_DEFAULT:", model)
print("facts:", len(facts_rows), "flights:", len(flight_rows))


OPENAI_API_KEY is set: True
MODEL_DEFAULT: gpt-5.4-nano
facts: 26 flights: 496


### A plain function, no model

Call it yourself first. This is the tool. The API does not know it exists until we describe it.


In [2]:
def get_fact(city):
    needle = city.strip().lower()
    for row in facts_rows:
        if row["City"].lower() == needle:
            return row["Fun Fact"]
    return "no fact for that city"


print(get_fact("Amsterdam"))


Amsterdam has more bicycles than people.


### First, the same question with no tools

Module 00 again. No schema, no `tools=`. Watch `finish_reason` and `content`. Whatever fact you get did not come from our CSV.


In [3]:
messages = [
    {"role": "user", "content": "Give me a fun fact about Amsterdam."}
]

response = client.chat.completions.create(
    model=model,
    messages=messages,
    max_completion_tokens=128,
    reasoning_effort="none",
)

message = response.choices[0].message
print("finish_reason:", response.choices[0].finish_reason)
print("content:      ", message.content)
print("tool_calls:   ", message.tool_calls)


finish_reason: stop
content:       Fun fact: Amsterdam’s canals are so central to the city that the famous “ring” of waterways is over 100 miles long in total—built in stages during the Dutch Golden Age.
tool_calls:    None


`finish_reason` is `stop`. `content` is a sentence. `tool_calls` is `None`. The model is guessing, the same way it guessed Prague weather.

Two names we will keep using:

- `messages` — the list we send on every `create`. Right now it is one user turn. A system prompt, if we had one, would sit in this list too.
- `message` — this one assistant reply. Today it is just text.

### The schema is a description, not a hook

A dict with a name, a description, and the arguments we expect. It does not bind Python. Nothing runs because this object exists.


In [4]:
get_fact_json = {
    "name": "get_fact",
    "description": "Look up a fun fact about a city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "City name, for example Amsterdam",
            }
        },
        "required": ["city"],
    },
}

tools = [{"type": "function", "function": get_fact_json}]


### Same question, now with `tools=tools`

Stop after the call. Do not run the function yet.


In [5]:
response = client.chat.completions.create(
    model=model,
    messages=messages,
    tools=tools,
    max_completion_tokens=128,
    reasoning_effort="none",
)

message = response.choices[0].message
print("finish_reason:", response.choices[0].finish_reason)
print("content:      ", message.content)
print("tool_calls:   ", message.tool_calls)


finish_reason: tool_calls
content:       None
tool_calls:    [ChatCompletionMessageFunctionToolCall(id='call_HP5oYzni9XxDf4MN6dKVrVki', function=Function(arguments='{"city":"Amsterdam"}', name='get_fact'), type='function')]


### What came back

`finish_reason` should now be `tool_calls`. `content` is often `None`. `message` is still one assistant reply — it just grew a new field. Nothing in our CSV was read. The function did not run. The model asked.


In [6]:
call = message.tool_calls[0]
print("id:        ", call.id)
print("name:      ", call.function.name)
print("arguments: ", call.function.arguments)
print("type of arguments:", type(call.function.arguments))


id:         call_HP5oYzni9XxDf4MN6dKVrVki
name:       get_fact
arguments:  {"city":"Amsterdam"}
type of arguments: <class 'str'>


`arguments` is a **string**. It looks like JSON. It is not a Python dict until we parse it.

```
assumed     LLM --> get_fact("Amsterdam")

actual      LLM --> {name, arguments-as-text, id}
            your code --> maybe get_fact(...)
```


### Now we decide to run it

Parse the string. Call the function we already tested. Print what we got. The model still has not seen this.


In [7]:
args = json.loads(call.function.arguments)
result = get_fact(args["city"])
print("parsed args:", args)
print("result:     ", result)


parsed args: {'city': 'Amsterdam'}
result:      Amsterdam has more bicycles than people.


### Send the result back

Three Python names, one more time:

- `messages` — the conversation we send. So far: the user question.
- `message` — the assistant turn that stopped for a tool (`tool_calls` on it).
- `result` — what *our* function returned. Not on the list yet.

The next `create` needs both new turns on `messages`: first `message` (the ask), then a `tool` turn with the same `id` and our `result`.


In [8]:
messages.append(message)
messages.append(
    {
        "role": "tool",
        "tool_call_id": call.id,
        "content": result,
    }
)

print("messages now has", len(messages), "turns:")
for turn in messages:
    role = turn["role"] if isinstance(turn, dict) else turn.role
    print(" -", role)


messages now has 3 turns:
 - user
 - assistant
 - tool


User, assistant (the ask), tool (our result). Now the model can talk.


In [9]:
final = client.chat.completions.create(
    model=model,
    messages=messages,
    tools=tools,
    max_completion_tokens=128,
    reasoning_effort="none",
)
print(final.choices[0].message.content)


Amsterdam has more bicycles than people.


That sentence was written from our CSV, not from the model's memory. If we had skipped the function and sent back a lie, the model would have repeated the lie. It cannot tell.

### A second tool

Same pattern. Flights live in the other CSV. Call it yourself first.


In [10]:
def get_flight(from_city, to_city):
    found = []
    a = from_city.strip().lower()
    b = to_city.strip().lower()
    for row in flight_rows:
        if row["from_city"].lower() == a and row["to_city"].lower() == b:
            found.append(row["price"] + " dollars, " + row["duration"] + " minutes")
    if not found:
        return "no flight found"
    return "; ".join(found)


print(get_flight("Sydney", "Madrid"))


249.66 dollars, 99 minutes


### The second schema, then one list

Each function gets its own description, the same shape as `get_fact_json`. Then we build `tools` from the two names. The list is what we send. The dicts are what we can read.


In [11]:
get_flight_json = {
    "name": "get_flight",
    "description": "Look up flights between two cities. Returns price in dollars and duration in minutes.",
    "parameters": {
        "type": "object",
        "properties": {
            "from_city": {"type": "string"},
            "to_city": {"type": "string"},
        },
        "required": ["from_city", "to_city"],
    },
}


In [12]:
tools = [
    {"type": "function", "function": get_fact_json},
    {"type": "function", "function": get_flight_json},
]

print([t["function"]["name"] for t in tools])


['get_fact', 'get_flight']


### One question that needs both

A flight from Sydney to Madrid, and a fun fact about Madrid. One reply may contain two `tool_calls`. We run whatever it asked, one after another.

If it only asks for one tool, we send that result back and `create` again. That small repeat is the seed of module 03. We cap it at three turns so it cannot run forever. We are not writing ReAct yet — no thoughts, no home-made verbs. Just: while it is still asking, we still run.


In [13]:
messages = [
    {
        "role": "user",
        "content": "How much is a flight from Sydney to Madrid, and give me a fun fact about Madrid.",
    }
]

for turn in range(3):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        max_completion_tokens=128,
        reasoning_effort="none",
    )
    message = response.choices[0].message
    print("turn", turn + 1, "finish_reason:", response.choices[0].finish_reason)
    print("n tool_calls:", len(message.tool_calls or []))

    if not message.tool_calls:
        print(message.content)
        break

    messages.append(message)
    for call in message.tool_calls:
        args = json.loads(call.function.arguments)
        print("asked:", call.function.name, args)
        if call.function.name == "get_fact":
            result = get_fact(args["city"])
        elif call.function.name == "get_flight":
            result = get_flight(args["from_city"], args["to_city"])
        else:
            result = "unknown tool"
        print("result:", result)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})


turn 1 finish_reason: tool_calls
n tool_calls: 2
asked: get_flight {'from_city': 'Sydney', 'to_city': 'Madrid'}
result: 249.66 dollars, 99 minutes
asked: get_fact {'city': 'Madrid'}
result: Madrid is home to the oldest restaurant in the world, Sobrino de Botín.


turn 2 finish_reason: stop
n tool_calls: 0
A flight from **Sydney to Madrid** costs **$249.66** and takes about **99 minutes**.

**Fun fact about Madrid:** Madrid is home to the **oldest restaurant in the world**, **Sobrino de Botín**.


The inner `for` is sequential on purpose: two calls in one `message`, run one after another. Module 07 is when those two become concurrent.

The outer `for turn in range(3)` is only there if the model asks for the flight in one reply and the fact in the next. Module 03 is that loop, named and kept.


### First look at the OpenAI Agents SDK

You just did the truth by hand: write a schema, parse `arguments`, run `get_fact`, append a `tool` message, call again.

A modern SDK does not replace that. It writes the schema from the function and runs your `for`. Two ideas today. Module 08 is the rest.

The package on PyPI is `openai-agents`. The import is `from agents import ...`. `pip install agents` is a different, older library. The course already has `openai-agents[viz]` from `uv sync`.

Talking points — say these once, then run:

1. Three names. `Agent` is instructions plus tools. `Runner` is the loop you wrote. `@function_tool` is your function, advertised.
2. Docs if you want them later: [openai.github.io/openai-agents-python](https://openai.github.io/openai-agents-python/).
3. Use `await Runner.run`. Jupyter already has an event loop; `run_sync` crashes here. Module 07 is why the SDK is async.
4. This is a first look. Module 08 is sessions, graphs, handoffs.

You typed `get_fact_json` by hand. `@function_tool` reads the signature and the docstring and builds that object. We wrap the `get_fact` you already wrote.


In [14]:
from agents import Agent, Runner, function_tool, trace, gen_trace_id


@function_tool
def fact_tool(city: str) -> str:
    """Look up a fun fact about a city."""
    return get_fact(city)


print(fact_tool)
print()
print("description:", fact_tool.description)
print()
print("params_json_schema:")
print(json.dumps(fact_tool.params_json_schema, indent=2))
print()
print("the dict you typed:")
print(json.dumps(get_fact_json["parameters"], indent=2))


FunctionTool(name='fact_tool', description='Look up a fun fact about a city.', params_json_schema={'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'fact_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x120ea8a50>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

description: Look up a fun fact about a city.

params_json_schema:
{
  "properties": {
    "city": {
      "title": "City",
      "type": "string"
    }
  },
  "required": [
    "city"
  ],
  "title": "fact_tool_args",
  "type": "object",
  "additionalProperties": false
}

the dict you typed:
{
  "type": "object",
  "properties": {
    "city": {


Same shape: a name, a description, `properties`, `required`. You stop typing JSON. The model still does not run Python. `Runner` still has to call `get_fact`.

Now the Amsterdam question, with the tool attached. `with trace("...")` records the run and prints a URL. Open it on the shared class key: [platform.openai.com/traces](https://platform.openai.com/traces).


In [15]:
amsterdam_agent = Agent(
    name="Facts",
    instructions="Use fact_tool. Do not invent a fact.",
    model=model,
    tools=[fact_tool],
)

trace_id = gen_trace_id()
print("Trace:", "https://platform.openai.com/traces/trace?trace_id=" + trace_id)

with trace("02 Amsterdam fact", trace_id=trace_id):
    sdk_out = await Runner.run(amsterdam_agent, "Give me a fun fact about Amsterdam.")
print(sdk_out.final_output)


Trace: https://platform.openai.com/traces/trace?trace_id=trace_9c815ab3d2d84b6789d9b0847398a2e3


Fun fact: Amsterdam has more bicycles than people.


### Now look at the trace

One agent span and one tool span named `fact_tool`. Same `name` + `arguments` you printed by hand, with a UI on it.

`final_output` is the sentence. You did not append a `tool` message. `Runner` did. `get_fact` is still **your** function. Sessions and `draw_graph` wait for module 08, when there is more than one agent to draw.


## 3. Observe

Two surfaces, same truth.

1. Dump the last assistant `message` from the hand-written loop. The protocol is small: a list of calls, each with an id, a name, and a string of arguments. Then print the `messages` list so the three kinds of turn are visible together.
2. Keep a traces tab open from the Amsterdam SDK cell. That UI is Observe for every framework from here on. Module 15 will grade traces. Today just find the `fact_tool` span.


In [16]:
if message.tool_calls:
    print(message.tool_calls[0].model_dump())
else:
    print("last message was text:")
    print(message.content)

print()
print("conversation:")
for turn in messages:
    if isinstance(turn, dict):
        role = turn["role"]
        extra = turn.get("content", "")
        if role == "tool":
            extra = (extra or "")[:80]
        elif role == "user":
            extra = (extra or "")[:80]
        else:
            extra = ""
    else:
        role = turn.role
        extra = "tool_calls" if turn.tool_calls else (turn.content or "")[:80]
    print(f"  {role:10} {extra}")


last message was text:
A flight from **Sydney to Madrid** costs **$249.66** and takes about **99 minutes**.

**Fun fact about Madrid:** Madrid is home to the **oldest restaurant in the world**, **Sobrino de Botín**.

conversation:
  user       How much is a flight from Sydney to Madrid, and give me a fun fact about Madrid.
  assistant  tool_calls
  tool       249.66 dollars, 99 minutes
  tool       Madrid is home to the oldest restaurant in the world, Sobrino de Botín.


Things to notice:

- `arguments` is still a string in the dump.
- `id` is how the model matches our later `tool` message. Mix them up and the next call fails.
- `messages` is the memory. `message` was one ask. `result` was our function.
- If the last `finish_reason` was `stop` and it never asked for a tool, it guessed. The schema is a request, not a guarantee.

You can refuse. Nothing forces you to run `get_flight` just because the model asked. That sentence is the start of module 14.


## 4. Challenge

Do one round trip for a city we have not used:

> Give me a fun fact about Istanbul.

Use the same `tools` list and the same `get_fact` function. When the model asks, you run the function. Then you send the result back and print the final sentence.

When you are done you should have:

- `asked_name` — the tool name on the first reply
- `fact` — the string `get_fact` returned
- `final_text` — the model's last `content`

The next cell only checks that those three exist and are not empty. It does not score the wording. We will look at a solution in the debrief.


In [ ]:
# asked_name, fact, final_text = ...


In [ ]:
assert asked_name, "asked_name should be the tool the model requested"
assert fact, "fact should be what get_fact returned"
assert final_text, "final_text should be the last model sentence"
print(asked_name)
print(fact)
print(final_text)
